In [ ]:
from google.colab import drive, files, output
import shutil

drive.mount('/content/drive')

client_secrets_path = '/content/drive/MyDrive/json/.client_secrets.json'
credentials_path = '/content/drive/MyDrive/json/.youtube-upload-credentials.json'
firebase_key_path = '/content/drive/MyDrive/json/.service_account_key.json'
shutil.copy(client_secrets_path, './client_secrets.json')
shutil.copy(credentials_path, './credentials.json')
shutil.copy(firebase_key_path, './service_account_key.json')
yt_client_secrets = './client_secrets.json'
yt_credentials = './credentials.json'
fb_service_account_key = './service_account_key.json'

In [7]:
!pip install PyCryptodome==3.17.0
!pip install yt_dlp
!pip install git+https://github.com/tokland/youtube-upload.git
!pip install firebase-admin
output.clear()

In [21]:
# built-in
import os
import cv2
import base64
import uuid
import numpy as np
import tempfile
import subprocess
import concurrent.futures

# 3rd party
import yt_dlp
from Crypto.Cipher import AES
from youtube_upload import auth, lib, upload_video
import firebase_admin
from firebase_admin import credentials as fb_credentials, firestore

In [22]:
home = os.path.expanduser("~")
yt_client_secrets = os.path.join(home, ".client_secrets.json")
yt_credentials = os.path.join(home, ".youtube-upload-credentials.json")
fb_service_account_key = os.path.join(home, ".service_account_key.json")

In [23]:
if not firebase_admin._apps:
    cred = fb_credentials.Certificate(fb_service_account_key)
    firebase_admin.initialize_app(cred)
db = firestore.client()

In [28]:
def quotient_remainder(divident, divsor):
    return divident // divsor, divident % divsor


def color_value(x):
    return x*255


def normal(x):
    return x/255


def encrypt_data_aes(data: bytes, key: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_EAX)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return base64.urlsafe_b64encode(cipher.nonce + tag + ciphertext)


def decrypt_data_aes(data: bytes, key: bytes) -> bytes:
    raw = base64.urlsafe_b64decode(data)
    nonce, tag, ciphertext = raw[:16], raw[16:32], raw[32:]

    cipher = AES.new(key, AES.MODE_EAX, nonce)
    clear_data = cipher.decrypt_and_verify(ciphertext, tag)
    return clear_data


def prepare_frame(args):
    frame_bytes, num_rows_per_frame, num_cols_per_frame, color_value, size = args
    frame_bits = np.unpackbits(frame_bytes)
    frame = color_value(frame_bits).reshape(num_rows_per_frame, num_cols_per_frame, 3).astype(np.uint8)
    newimg = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
    return newimg


def detect_ffmpeg_gpu_encoder():
    # Check for NVIDIA GPU
    try:
        result = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True)
        encoders = result.stdout
        if 'hevc_nvenc' in encoders:
            return 'hevc_nvenc'
        elif 'hevc_qsv' in encoders:
            return 'hevc_qsv'
        elif 'hevc_videotoolbox' in encoders:
            return 'hevc_videotoolbox'
    except Exception:
        pass
    return 'libx265'  # CPU fallback


def encode(infile_path, outvideo_path, encrypt, key,
           fps=20, num_cols_per_frame=64, num_rows_per_frame=36):
    with open(infile_path, 'rb') as fd:
        raw_data_bytes = fd.read()
    if encrypt:
        raw_data_bytes = encrypt_data_aes(raw_data_bytes, key)
    data_bytes = np.frombuffer(raw_data_bytes, dtype=np.uint8)
    len_of_data = len(data_bytes)
    num_bytes_per_row = int(num_cols_per_frame * 3 / 8)
    num_bytes_per_frame = num_bytes_per_row * num_rows_per_frame

    len_bytes = np.frombuffer(len_of_data.to_bytes(8, byteorder='big'), dtype=np.uint8)
    total_data = [len_bytes, data_bytes]

    (num_frames, num_leftover_bytes) = quotient_remainder(8 + len_of_data, num_bytes_per_frame)

    if num_leftover_bytes > 0:
        num_bytes_last_frame_padding = num_bytes_per_frame - num_leftover_bytes
        padding_bytes = np.zeros(num_bytes_last_frame_padding, dtype=np.uint8)
        total_data.append(padding_bytes)
        num_frames += 1

    data_bytes = np.concatenate(total_data)

    size = (num_cols_per_frame * 20, num_rows_per_frame * 20)

    args_list = [
        (
            data_bytes[i * num_bytes_per_frame: (i + 1) * num_bytes_per_frame],
            num_rows_per_frame,
            num_cols_per_frame,
            color_value,
            size
        )
        for i in range(num_frames)
    ]

    # Create a temporary directory for PNG frames
    with tempfile.TemporaryDirectory() as temp_dir:
        # Save frames as PNG images in the temp directory
        with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
            for idx, newimg in enumerate(executor.map(prepare_frame, args_list)):
                cv2.imwrite(os.path.join(temp_dir, f"frame_{idx:04d}.png"), newimg)

        # Detect GPU encoder or fallback to CPU
        encoder = detect_ffmpeg_gpu_encoder()
        print(f"Using FFmpeg encoder: {encoder}")

        # Run ffmpeg using images from the temp directory
        ffmpeg_cmd = (
            f'ffmpeg -y -framerate {fps} -i "{os.path.join(temp_dir, "frame_%04d.png")}" '
            f'-c:v {encoder} "{outvideo_path}"'
        )
        os.system(ffmpeg_cmd)


def process_frame(args):
    frame, step = args
    blocks = frame.reshape(frame.shape[0]//step, step, frame.shape[1]//step, step, 3)
    blocks = blocks.transpose(0,2,1,3,4).reshape(-1, step*step, 3)
    means = normal(blocks.mean(axis=1)).round().astype(np.uint8)
    return means


def decode(invideo_path, outfile_path, decrypt, key):
    step = 20
    cap = cv2.VideoCapture(invideo_path)
    data_bits_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        data_bits_list.append(process_frame((frame, step)))
    cap.release()
    data_bits = np.concatenate(data_bits_list).reshape(-1, 1)
    data_bytes = np.packbits(data_bits)
    len_of_data = int.from_bytes(data_bytes[:8], byteorder='big')
    data_bytes_retrieved = data_bytes[8:len_of_data+8].tobytes()
    if decrypt:
        data_bytes_retrieved = decrypt_data_aes(data_bytes_retrieved, key)
    with open(outfile_path, 'wb') as fd:
        fd.write(data_bytes_retrieved)


In [26]:
def get_youtube_upload_handler():
    """Return the API Youtube object."""
    # home = os.path.expanduser("~")
    # client_secrets = os.path.join(home, ".client_secrets.json")
    # credentials = os.path.join(home, ".youtube-upload-credentials.json")
    get_code_callback = (auth.console.get_code)
    return auth.get_resource(yt_client_secrets, yt_credentials, get_code_callback=get_code_callback)


def upload_youtube_video(youtube, title, video_path):
    """Upload video with index (for split videos)."""
    u = lib.to_utf8
    title = u(title)

    request_body = {
        "snippet": {
            "title": title,
            "description": "",
            "categoryId": None,
            "tags": u(""),
            "defaultLanguage": None,
            "defaultAudioLanguage": None

        },
        "status": {
            "embeddable": True,
            "privacyStatus": "unlisted",
            "publishAt": None,
            "license": "youtube",
        },
        "recordingDetails": {
            "location": None,
            "recordingDate": None,
        },
    }

    return upload_video.upload(youtube, video_path, request_body)


def download_youtube_video(video_id, output_file_path):
    ydl_opts = {
        'outtmpl': output_file_path,
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4',
        'merge_output_format': 'mp4',
        'verbose': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([f"https://www.youtube.com/watch?v={video_id}"])

def get_video_id_by_filename(filename):
    safe_filename = filename.replace("/", "_")
    doc = db.collection('uploads').document(safe_filename).get()
    if doc.exists:
        return doc.to_dict()['video_id']
    else:
        print("No video ID found for this filename.")
        return None

def youtube_upload(upload_file_path, encrypt, key, video_fps):
    video_path = tempfile.mktemp(".mp4")
    encode(upload_file_path, video_path, encrypt, key, video_fps)
    youtube = get_youtube_upload_handler()
    video_id = upload_youtube_video(youtube, f"DATA-{str(uuid.uuid4()).upper()}", video_path)
    os.remove(video_path)
    print(f"YouTube Video ID: {video_id}", f"YouTube: https://youtu.be/{video_id}", sep="\n")
    filename = os.path.basename(upload_file_path)
    safe_filename = filename.replace("/", "_")
    db.collection('uploads').document(safe_filename).set({'video_id': video_id})

def youtube_retrieve(video_id, output_file_path, decrypt, key):
    video_path = tempfile.mktemp(".mp4")
    print(video_path)
    download_youtube_video(video_id, video_path)
    decode(video_path, output_file_path, decrypt, key)


In [ ]:
# upload

uploaded = files.upload()
i = os.path.join("/content/", next(iter(uploaded.keys())))
encrypt = False
key = ""

key = str(key).encode("ascii")[:16] if encrypt and key is not None else ""
youtube_upload(i, encrypt, key, video_fps=20)


Saving 1MB.jpg to 1MB.jpg
Using FFmpeg encoder: hevc_nvenc
YouTube Video ID: yDuEvptkbiA
YouTube: https://youtu.be/yDuEvptkbiA


In [29]:
youtube_upload("3283375.zip", False, "", video_fps=20)

: 

In [ ]:
# retrieve

video_id = input("Enter YouTube Video ID to retrieve: ")
output_file_path = input("Enter output file path to save the retrieved video: ")
decrypt = False
key = ""

key = str(key).encode("ascii")[:16] if decrypt and key is not None else ""
youtube_retrieve(video_id, output_file_path, decrypt, key)

In [19]:
# retrieve from filename

output_file_path = input("Enter output file path: ")
filename = os.path.basename(output_file_path)
video_id = get_video_id_by_filename(filename)
if video_id:
    decrypt = False
    key = ""
    key = str(key).encode("ascii")[:16] if decrypt and key is not None else ""
    youtube_retrieve(video_id, output_file_path, decrypt, key)

[debug] Encodings: locale UTF-8, fs utf-8, pref UTF-8, out UTF-8 (No ANSI), error UTF-8 (No ANSI), screen UTF-8 (No ANSI)
[debug] yt-dlp version stable@2025.05.22 from yt-dlp/yt-dlp [7977b329e] (pip) API
[debug] params: {'outtmpl': '/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.mp4', 'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/mp4', 'merge_output_format': 'mp4', 'verbose': True, 'compat_opts': set(), 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.20 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-us,en;q=0.5', 'Sec-Fetch-Mode': 'navigate'}}
[debug] Python 3.10.11 (CPython arm64 64bit) - macOS-15.5-arm64-arm-64bit (OpenSSL 3.5.0 8 Apr 2025)


/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.mp4


[debug] exe versions: ffmpeg 7.1.1 (setts), ffprobe 7.1.1
[debug] Optional libraries: Cryptodome-3.21.0, brotli-1.1.0, certifi-2025.04.26, requests-2.31.0 (unsupported), sqlite3-3.43.2, urllib3-2.4.0, websockets-15.0.1
[debug] Proxy map: {}
[debug] Request Handlers: urllib, websockets
[debug] Plugin directories: none
[debug] Loaded 1859 extractors
[debug] [youtube] [pot] PO Token Providers: none
[debug] [youtube] [pot] PO Token Cache Providers: memory
[debug] [youtube] [pot] PO Token Cache Spec Providers: webpo


[youtube] Extracting URL: https://www.youtube.com/watch?v=yDuEvptkbiA
[youtube] yDuEvptkbiA: Downloading webpage
[youtube] yDuEvptkbiA: Downloading tv client config
[youtube] yDuEvptkbiA: Downloading player 3b4b7883-main


[debug] Saving youtube-sts.3b4b7883-main to cache


...

[download] Destination: /var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f136.mp4
[download] 100% of   47.35MiB in 00:00:25 at 1.83MiB/s   


[debug] Invoking http downloader on "https://rr4---sn-c0q7lnse.googlevideo.com/videoplayback?expire=1748909660&ei=_Ok9aMzvKL7g7OsPoMvCwAs&ip=104.28.230.10&id=o-ANW1QuBevxp2jU4UAyS_RRS5Go4uEFouXiPeM55Q8td7&itag=140&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&met=1748888060%2C&mh=KS&mm=31%2C29&mn=sn-c0q7lnse%2Csn-5goeenez&ms=au%2Crdu&mv=m&mvi=4&pl=24&rms=au%2Cau&initcwndbps=2083750&bui=AY1jyLNV3i-YQFBhv3WPuymSY4eW35oBDku5rnS2pmfzcf5rAp1XN0L6KuB17EO5_c_733q6YOchtRXo&vprv=1&svpuc=1&mime=audio%2Fmp4&ns=zgEuPrUwBkhhuzr6W29scZAQ&rqh=1&gir=yes&clen=965408&dur=59.605&lmt=1748883950667460&mt=1748887534&fvip=1&keepalive=yes&lmw=1&fexp=51466698&c=TVHTML5&sefc=1&txp=6208224&n=iKpq9INmv6wdjg&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cbui%2Cvprv%2Csvpuc%2Cmime%2Cns%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AJfQdSswRQIgcGiz7q4zuvJCga4gX6kRPTn5pVbvHzuxmthlbVk5Rn4CIQDHKh7tk29gZUMXPqZQieAuW38wUR6C6R6ul-g4Gu64Mw%3D%3D&lsparams=met%2Cmh%2Cmm%2Cmn%2Cms%2Cmv%2Cmvi%2Cpl%2Crms%2Cinitcwn

[download] Destination: /var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f140.m4a
[download] 100% of  942.78KiB in 00:00:03 at 263.80KiB/s 
[Merger] Merging formats into "/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.mp4"


[debug] ffmpeg command line: ffmpeg -y -loglevel repeat+info -i file:/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f136.mp4 -i file:/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f140.m4a -c copy -map 0:v:0 -map 1:a:0 -movflags +faststart file:/var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.temp.mp4


Deleting original file /var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f136.mp4 (pass -k to keep)
Deleting original file /var/folders/dw/9k4z_1x15v7fh4gt9sb2whfm0000gn/T/tmp2s25flkr.f140.m4a (pass -k to keep)


In [ ]:
# encode test
encode("1MB.jpg", "1MB.mp4", False, "")

Using FFmpeg encoder: hevc_videotoolbox


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex